# From paths to an experiment estimate
## Retentioneering · assignment populations, useful outcomes and honest uncertainty

**Product question:** does a changed cart improve purchasing without increasing checkout backtracking?
We will connect Retentioneering paths to explicit outcomes, preserve silent assignments, estimate
effects and uncertainty, and distinguish three different questions that often get called conversion.

The first example is an **independent, predeclared event simulation** with 500 users per arm.
Its fixed seed is not searched for a desirable result. Small hand-computable examples then expose
ratio and repeated-session traps. The complete local edition finally applies the same workflow
to the frozen, selected Lumeo teaching case, where sparse outcomes expose useful failure states.

Native Retentioneering handles streams, graphs, step matrices, segments and path metrics.
The development branch adds native `stream.experiment_data` / `stream.experiment` APIs.
Install the accompanying source branch: these methods are not in the PyPI 5.3.0 release.
The notebook adapter turns a saved payload into tables and plots; it performs no inference.

In [ ]:
import os

os.environ["RETENTIONEERING_NO_TRACK"] = "1"
import hashlib
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import retentioneering as rete
from IPython.display import display
from lumeo_data import load_lumeo_tables
from retentioneering.tools.experiment import session_funnel_data
from experiment_plots import plot_effects, plot_funnel
from experiment_demo import (
    CATALOG,
    END,
    OBSERVATION,
    ORDER,
    ORDERS,
    PATTERN,
    REVENUE,
    SCHEMA,
    START,
    STEPS,
    make_demo_dataset,
    make_ratio_fixture,
    make_repeated_sessions,
    manual_ordered_reach,
)
from experiment_helpers import result_from_payload, save_figure, show_native

ROOT = Path.cwd()
OUT = Path(os.environ.get("LUMEO_AB_OUTPUT_DIR", "artifacts"))
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "figure.dpi": 110})
BOOT = {
    "method": "percentile",
    "n_resamples": 4999,
    "confidence_level": 0.95,
    "random_state": 42,
    "batch_size": 128,
}
BINARY = {"method": "newcombe", "confidence_level": 0.95}
print("Retentioneering", rete.__version__)

## 1. Specify the experiment before reading paths

Every user is assigned before a common `[2 January, 16 January)` outcome window. Assignment
is balanced: 500 control and 500 treatment users. A separate roster includes everyone.
All have prehistory; some have no events in the outcome window. A last event is not a censoring time.

The generator's purchase probabilities are **.10 and .13**. Buyers have `1 + Poisson(1.2)` orders,
with mean order prices **GBP 80 and 88**. Users can make multiple visits. This deliberately simple
data-generating process supplies known population quantities, not claims about a real product.

The declared primary outcome is purchase per assigned user. Revenue per assigned user is secondary.
Backtracking is a **descriptive/exploratory guardrail candidate**: a pattern found after looking
at paths would need a later, predeclared experiment for confirmatory interpretation.

We keep the first fixed-seed realization even if its estimate differs substantially from the
population effect. A single nominal 95% interval is not guaranteed to contain that effect.

In [ ]:
demo = make_demo_dataset(n_per_arm=500, seed=20261029)
stream = rete.Eventstream(demo["events"], schema=SCHEMA)
window = stream.filter_events(
    func=lambda df: df.timestamp.ge(START) & df.timestamp.lt(END)
)
population = (
    demo["unit_truth"]
    .groupby("variant")
    .agg(
        assigned=("user_id", "size"),
        observed=("active", "sum"),
        buyers=("converted", "sum"),
        orders=("n_orders", "sum"),
        sessions=("n_sessions", "sum"),
    )
)
population["silent"] = population.assigned - population.observed
display(population)
display(
    pd.Series(
        demo["metadata"]["population_truth"], name="DGP target, not realized estimate"
    ).to_frame()
)
assert len(demo["assignments"]) == 1000
assert (population.assigned == population.observed + population.silent).all()

## 2. Use paths to define an interpretable outcome

The native graph and step matrix show **observed session paths around checkout**. They condition
on reaching checkout and cannot estimate the overall assignment effect. Product-view repetitions
are collapsed on a visual branch only; all outcome calculations use the original stream.

A return from checkout to cart may represent editing, uncertainty or a usability problem. The
pattern `checkout_start → cart → checkout_start` is therefore a measurable behavior, not proof
of frustration or a causal mediator. In this fixture it has the same meaning in both variants.

In [ ]:
visual = window.collapse_events(loops=["product_view"])
graph = visual.transition_graph(
    path_col="session_id",
    path_pattern="checkout_start",
    diff=("variant", "treatment", "control"),
    edge_weight="unique_paths",
    height=440,
)
show_native(
    graph,
    OUT,
    "demo_paths.html",
    "Observed checkout sessions",
    "Treatment minus control. Descriptive session paths; not the assignment population.",
)
matrix = visual.step_matrix(
    path_col="session_id",
    anchor="checkout_start",
    step_window=3,
    diff=("variant", "treatment", "control"),
    height=440,
)
show_native(
    matrix,
    OUT,
    "demo_step_matrix.html",
    "Around the first checkout start",
    "A backward cart transition suggests a behavioral outcome to investigate, not a proven mechanism.",
)

## 3. Reuse a native path metric in the experiment

The same `matches_pattern` definition feeds native `get_metrics` and the new experiment calculation.
The arrows below mean **adjacent** events. At user grain the path can include multiple sessions;
this fixture's backtracking sequence occurs inside a session. If a real outcome must be limited
to one session, its path/window contract must say so explicitly.

`segment_overview_data` lets us describe the pattern by variant. Its observed-user denominator
is useful for diagnosis. The experiment completes that metric from the assignment roster so a
silent user has the structurally defined zero flag. No outcome-based segment replaces the roster.

In [ ]:
BACKTRACK = {"metric": "matches_pattern", "metric_args": {"pattern": PATTERN}}
display(window.get_metrics([ORDER, BACKTRACK], path_col="user_id").head())
display(
    window.segment_overview_data(
        segment_col="variant",
        path_col="user_id",
        metrics=[{**BACKTRACK, "agg": "mean"}, {**ORDER, "agg": "mean"}],
    )
)
metrics = [
    {
        "name": "conversion",
        **ORDER,
        "agg": "mean",
        "display": {
            "label": "Purchase per assigned user",
            "unit": "proportion",
            "scale": 100,
            "role": "primary",
            "direction": "higher_is_better",
        },
    },
    {
        "name": "revenue_per_user",
        **REVENUE,
        "agg": "mean",
        "display": {
            "label": "Revenue per assigned user",
            "unit": "GBP / assigned user",
            "role": "secondary",
            "direction": "higher_is_better",
        },
    },
    {
        "name": "average_order_value",
        "agg": "ratio_of_sums",
        "numerator": REVENUE,
        "denominator": ORDERS,
        "display": {
            "label": "Pooled average order value",
            "unit": "GBP / order",
            "role": "secondary",
            "direction": "higher_is_better",
        },
    },
    {
        "name": "checkout_backtrack",
        **BACKTRACK,
        "agg": "mean",
        "display": {
            "label": "Checkout backtracking",
            "unit": "proportion",
            "scale": 100,
            "role": "guardrail",
            "direction": "lower_is_better",
        },
    },
]

## 4. Estimate the declared quantities

Purchase and backtracking are binary outcomes on independent assigned users. We explicitly choose
**Wilson arm intervals and a Newcombe difference interval** for those binary proportions, including
all-zero outcomes. Revenue and AOV use **whole-user percentile bootstrap**. These are different
interval procedures; the result records the method for each estimate.

Every bootstrap replica gives one weight to a person's complete contribution vector: all orders,
visits, numerator/denominator pairs and funnel steps. AOV recomputes total revenue / total orders;
revenue per user keeps all assignments in its denominator. Every interval is pointwise.

Fisher's exact test and Welch's mean test are separately declared, with Holm over two outcomes.
Their p-values are not bootstrap tail probabilities. The balanced allocation design is recorded;
we do not attach a Bernoulli-allocation SRM test to this fixed-count randomization.

In [ ]:
result = result_from_payload(
    stream.experiment_data(
        assignments=demo["assignments"],
        unit_col="user_id",
        variant_col="variant",
        observation=OBSERVATION,
        metrics=metrics,
        event_catalog=CATALOG,
        bootstrap=BOOT,
        binary_intervals=BINARY,
        funnel_steps=STEPS,
        tests={
            "methods": {"conversion": "fisher_exact", "revenue_per_user": "welch"},
            "adjustment": "holm",
        },
    )
)
score = result.summary.loc[result.summary.family.eq("scorecard")].set_index("metric")
display(result.diagnostics)
display(
    score[
        [
            "control",
            "treatment",
            "absolute_effect",
            "absolute_effect_ci_low",
            "absolute_effect_ci_high",
            "absolute_effect_ci_method",
            "absolute_effect_status",
        ]
    ]
)

In [ ]:
experiment_widget = stream.experiment(
    assignments=demo["assignments"],
    unit_col="user_id",
    variant_col="variant",
    observation=OBSERVATION,
    metrics=metrics,
    event_catalog=CATALOG,
    bootstrap=BOOT,
    binary_intervals=BINARY,
    funnel_steps=STEPS,
    tests={
        "methods": {"conversion": "fisher_exact", "revenue_per_user": "welch"},
        "adjustment": "holm",
    },
)
show_native(
    experiment_widget,
    OUT,
    "experiment_native.html",
    "Native experiment result",
    "Same specification, seed and aggregate payload as the headless calculation.",
)
assert json.loads(experiment_widget.result) == result.to_dict()

## 5. Read effect, uncertainty and support together

Percentage points and GBP must not share a numeric axis. The interval belongs to the **difference
itself**, not to the subtraction of two marginal CI endpoints. Check outcome support as well as
assignment N: hundreds of assigned users with only a few positive values do not make a stable
revenue distribution. Relative uplift is secondary and can be undefined at a zero control baseline.

The DGP target is known only because this is a simulation. We show it alongside the realized
estimate without changing the seed or choosing a different method when an interval misses it.

In [ ]:
fig = plot_effects(
    result,
    ["conversion", "revenue_per_user", "average_order_value"],
    ["Purchase per assigned user", "Revenue per assigned user", "Pooled order value"],
    ["percentage points", "GBP / assigned user", "GBP / order"],
    scales=[100, 1, 1],
)
save_figure(fig, OUT, "demo_effects")
plt.show()
support_columns = [
    "control_units",
    "treatment_units",
    "control_nonzero_units",
    "treatment_nonzero_units",
    "control_denominator",
    "treatment_denominator",
]
display(score[[c for c in support_columns if c in score.columns]])
conversion = score.loc["conversion"]
target = demo["metadata"]["population_truth"]["conversion_effect"]
display(
    pd.DataFrame(
        [
            {
                "DGP_effect_pp": 100 * target,
                "realized_effect_pp": 100 * conversion.absolute_effect,
                "CI_low_pp": 100 * conversion.absolute_effect_ci_low,
                "CI_high_pp": 100 * conversion.absolute_effect_ci_high,
                "this_interval_covers_DGP_target": conversion.absolute_effect_ci_low
                <= target
                <= conversion.absolute_effect_ci_high,
            }
        ]
    )
)

## 6. Keep the funnel's question visible

Cumulative ordered reach divides by **all assigned users**. A purchase indicator asks whether an
order occurs; funnel completion additionally requires the entire sequence in order inside the
window. At user grain those steps can span visits.

Conditional transitions divide by a previous reached step, whose population treatment may change.
For example, with 100 assigned users per arm, control can have 8 buyers among 10 reaching checkout,
and treatment 20 buyers among 50: assigned conversion rises by 12 pp while conditional conversion
falls by 40 pp. The second contrast alone does not establish a worse checkout mechanism.

In [ ]:
fig = plot_funnel(result, "funnel_cumulative", "Ordered reach / all assigned users")
save_figure(fig, OUT, "demo_assigned_funnel")
plt.show()
fig = plot_funnel(
    result, "funnel_conditional", "Conditional transitions: previous-step denominators"
)
save_figure(fig, OUT, "demo_conditional_funnel")
plt.show()
display(
    result.summary.loc[
        result.summary.family.eq("funnel_conditional"),
        [
            "step",
            "control_denominator",
            "treatment_denominator",
            "absolute_effect_ci_method",
            "absolute_effect_status",
        ],
    ]
)

## 7. Audit a few people's contributions

The calculation is easier to trust when it can be traced from a silent person, an active nonbuyer
and a repeat buyer to the totals. The independent generator truth table below is used only as an
audit oracle. It is never passed as the observed outcomes to the experiment estimator.

The literal Python sequence scan checks the funnel without reusing the library's SQL. Aggregate
JSON exports omit user identifiers; this local audit table is intentionally separate.

In [ ]:
truth = demo["unit_truth"].set_index("user_id")
audit_ids = (
    list(truth.query("active == 0").head(1).index)
    + list(truth.query("active == 1 and converted == 0").head(1).index)
    + list(truth.query("n_orders > 1").head(2).index)
)
display(
    truth.loc[
        audit_ids,
        [
            "variant",
            "active",
            "n_sessions",
            "n_orders",
            "revenue",
            "checkout_backtrack",
        ],
    ]
)
native_unit_metrics = window.get_metrics(
    [ORDER, ORDERS, REVENUE, BACKTRACK], path_col="user_id"
)
native_unit_metrics.columns = ["converted", "n_orders", "revenue", "checkout_backtrack"]
unit_audit = (
    demo["assignments"].set_index("user_id")[["variant"]].join(native_unit_metrics)
)
unit_audit[native_unit_metrics.columns] = unit_audit[
    native_unit_metrics.columns
].fillna(0.0)
display(unit_audit.loc[audit_ids])
manual = truth.groupby("variant").agg(
    conversion=("converted", "mean"),
    revenue_per_user=("revenue", "mean"),
    numerator=("revenue", "sum"),
    denominator=("n_orders", "sum"),
    checkout_backtrack=("checkout_backtrack", "mean"),
)
manual["average_order_value"] = manual.numerator / manual.denominator
for name in [
    "conversion",
    "revenue_per_user",
    "average_order_value",
    "checkout_backtrack",
]:
    assert np.isclose(score.loc[name, "control"], manual.loc["control", name])
    assert np.isclose(score.loc[name, "treatment"], manual.loc["treatment", name])
raw_window = demo["events"].loc[
    demo["events"].timestamp.ge(START) & demo["events"].timestamp.lt(END)
]
scan = manual_ordered_reach(raw_window, STEPS).reindex(truth.index, fill_value=0)
for arm in ["control", "treatment"]:
    expected = scan.loc[truth.index[truth.variant.eq(arm)]].mean().to_numpy()
    actual = result.summary.loc[
        result.summary.family.eq("funnel_cumulative"), arm
    ].to_numpy()
    assert np.allclose(expected, actual)
print("Points agree with independent per-user and sequence-scan oracles.")

## 8. Two averages can legitimately point in opposite directions

Consider two customers per arm. Control contributes `(GBP 10, 1 order)` and `(GBP 90, 9 orders)`;
treatment contributes `(GBP 15, 1 order)` and `(GBP 72, 9 orders)`.

The pooled average gives each **order** equal weight: 10 versus 8.7, difference **−1.3 GBP**.
The mean personal average gives each eligible **person** equal weight: 10 versus 11.5, difference
**+1.5 GBP**. We calculate both through the proposed API. Four people are a hand-computable
estimand counterexample, not an inferential claim of a real effect.

In [ ]:
ratio_case = make_ratio_fixture()
ratios = result_from_payload(
    rete.Eventstream(ratio_case["events"], schema=SCHEMA).experiment_data(
        assignments=ratio_case["assignments"],
        variant_col="variant",
        observation=OBSERVATION,
        event_catalog=CATALOG,
        metrics=[
            {
                "name": "pooled_aov",
                "agg": "ratio_of_sums",
                "numerator": REVENUE,
                "denominator": ORDERS,
            },
            {
                "name": "mean_personal_aov",
                "agg": "mean_of_ratios",
                "numerator": REVENUE,
                "denominator": ORDERS,
                "eligibility": "denominator_positive",
            },
        ],
    )
)
display(ratios.summary[["metric", "control", "treatment", "absolute_effect"]])
assert np.allclose(ratios.summary.absolute_effect, [-1.3, 1.5])

## 9. More visits are not more independent people

Eight people contribute one session each: control has 2 buyers out of 4, treatment 3 out of 4.
Now repeat each person's identical session 100 times. The session-weighted difference is still
25 pp. User-cluster uncertainty must also remain unchanged: there is no new independent person.

Each visit has exactly one cart opportunity. We estimate total successful orders / total cart
opportunities, using the experiment's independent within-arm user draws. The denominator is visits;
the sampling unit is still people. Eight people give a wide interval. The invariance check is a
correctness property, not evidence about population coverage.

In [ ]:
repeat_results = []
for repeats in [1, 100]:
    repeat_frame = make_repeated_sessions(repeats)
    repeat_roster = (
        repeat_frame[["user_id", "variant"]]
        .drop_duplicates()
        .assign(assigned_at=pd.Timestamp("2026-01-01"))
    )
    calculated = result_from_payload(
        rete.Eventstream(repeat_frame, schema=SCHEMA).experiment_data(
            assignments=repeat_roster,
            variant_col="variant",
            observation=OBSERVATION,
            event_catalog=CATALOG,
            metrics=[
                {
                    "name": "session_completion",
                    "agg": "ratio_of_sums",
                    "numerator": ORDERS,
                    "denominator": {
                        "metric": "event_count",
                        "metric_args": {"event": "add_to_cart"},
                    },
                }
            ],
            bootstrap={**BOOT, "n_resamples": 1999, "random_state": 73},
        )
    )
    row = calculated.summary.iloc[0]
    repeat_results.append(
        {
            "sessions_per_person": repeats,
            "independent_people": 8,
            "effect_pp": 100 * row.absolute_effect,
            "CI_low_pp": 100 * row.absolute_effect_ci_low,
            "CI_high_pp": 100 * row.absolute_effect_ci_high,
        }
    )
display(pd.DataFrame(repeat_results))
assert np.allclose(
    [repeat_results[0][k] for k in ["effect_pp", "CI_low_pp", "CI_high_pp"]],
    [repeat_results[1][k] for k in ["effect_pp", "CI_low_pp", "CI_high_pp"]],
)

## 10. A valid zero is not a claim of certainty

The observed test family and interval methods are explicit. Fisher tests a binary outcome;
Welch tests a user-level mean. Holm addresses the two declared tests, not every exploratory
graph or segment that might later be inspected. No observed-power field or automatic winner is used.

For a boundary illustration, remove all observed orders from a copy while retaining the declared
event catalogue and assignments. The binary point estimates are zero. Wilson/Newcombe still
express sampling uncertainty; Fisher's p-value of 1 does not establish equivalence. This is an
artificial boundary check, not a corrected version of the original data.

In [ ]:
display(score[["p_value", "p_value_adjusted", "test_method", "test_status"]])
zero_stream = rete.Eventstream(
    demo["events"].loc[demo["events"].event.ne("order_success")].copy(), schema=SCHEMA
)
boundary = result_from_payload(
    zero_stream.experiment_data(
        assignments=demo["assignments"],
        variant_col="variant",
        observation=OBSERVATION,
        event_catalog=CATALOG,
        metrics=[{"name": "conversion", **ORDER, "agg": "mean"}],
        binary_intervals=BINARY,
        tests={"methods": {"conversion": "fisher_exact"}, "adjustment": "holm"},
    )
)
display(
    boundary.summary[
        [
            "control",
            "treatment",
            "absolute_effect",
            "absolute_effect_ci_low",
            "absolute_effect_ci_high",
            "absolute_effect_ci_method",
            "p_value",
            "relative_effect_status",
        ]
    ]
)

## 11. Transfer the same workflow to frozen Lumeo

Lumeo v8.1E3.5 is a **selected synthetic teaching case**. The 4,000-user frozen subset was chosen
without outcomes, but that does not undo selection of its parent. We use it to show integration
and limits, not to validate CI coverage or establish a real product effect.

The question is purchase in `[11 May, 25 May)` among users assigned before 11 May. This late
calendar-window outcome is not 14-day conversion since each assignment. Events have no variant
column: assignments provide it. The simulation contract supplies complete event observation;
screen recording completeness is a separate layer.

In [ ]:
lumeo = load_lumeo_tables(ROOT / "data" / "lumeo")
input_hashes = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in (ROOT / "data" / "lumeo").glob("*")
    if p.is_file()
}
LSTART, LEND = (pd.Timestamp("2026-05-11"), pd.Timestamp("2026-05-25"))
roster = lumeo["assignments"].loc[lumeo["assignments"].assigned_at < LSTART].copy()
raw = (
    lumeo["events"]
    .loc[lumeo["events"].user_id.isin(roster.user_id)]
    .merge(roster[["user_id", "exp_group"]], on="user_id", validate="many_to_one")
)
lumeo_stream = rete.Eventstream(
    raw,
    schema={
        "path_cols": ["user_id", "session_id"],
        "segment_cols": ["exp_group", "device", "platform"],
        "custom_cols": ["order_value_gbp", "order_items"],
    },
)
lumeo_window = lumeo_stream.filter_events(
    func=lambda df: df.timestamp.ge(LSTART) & df.timestamp.lt(LEND)
)
in_window = raw.loc[raw.timestamp.ge(LSTART) & raw.timestamp.lt(LEND)]
lumeo_population = roster.groupby("exp_group").size().rename("assigned").to_frame()
lumeo_population["active"] = (
    in_window.groupby("exp_group")
    .user_id.nunique()
    .reindex(lumeo_population.index, fill_value=0)
)
lumeo_population["silent"] = lumeo_population.assigned - lumeo_population.active
lumeo_population["sessions"] = (
    in_window.groupby("exp_group")
    .session_id.nunique()
    .reindex(lumeo_population.index, fill_value=0)
)
display(lumeo_population)

## 12. One event log, four different questions

The table below places four estimands beside one another. The active-user denominator is a
conditional description. Session completion uses a different path grain as well as a different
denominator. These estimates can differ in sign without a calculation error.

The assignment population must remain intact for the assigned-user outcome. User-level ordered
reach can connect steps across sessions, whereas session reach requires them in the same session.

In [ ]:
lumeo_result = result_from_payload(
    lumeo_stream.experiment_data(
        assignments=roster,
        variant_col="exp_group",
        observation={
            "start": LSTART.isoformat(),
            "end": LEND.isoformat(),
            "assignment_time_col": "assigned_at",
            "complete": True,
        },
        event_catalog={"event": sorted(lumeo["events"].event.unique())},
        metrics=metrics[:3],
        funnel_steps=STEPS,
        bootstrap=BOOT,
        binary_intervals=BINARY,
    )
)
lumeo_sessions = session_funnel_data(
    lumeo_window, steps=STEPS, segment_col="exp_group", bootstrap=BOOT
)
buyers = (
    in_window.loc[in_window.event.eq("order_success")]
    .groupby("exp_group")
    .user_id.nunique()
)
ls = lumeo_result.summary.set_index("metric")
ss = lumeo_sessions.summary.set_index("metric")
questions = pd.DataFrame(
    {
        "Purchase / assigned": [
            ls.loc["conversion", "control"],
            ls.loc["conversion", "treatment"],
        ],
        "Purchase / active": [
            buyers["control"] / lumeo_population.loc["control", "active"],
            buyers["treatment"] / lumeo_population.loc["treatment", "active"],
        ],
        "Ordered user completion / assigned": [
            ls.loc["funnel_cumulative_4", "control"],
            ls.loc["funnel_cumulative_4", "treatment"],
        ],
        "Ordered session completion / sessions": [
            ss.loc["funnel_cumulative_4", "control"],
            ss.loc["funnel_cumulative_4", "treatment"],
        ],
    },
    index=["control", "treatment"],
).T
questions["difference_pp"] = 100 * (questions.treatment - questions.control)
display(questions)
native = lumeo_window.funnel_data(
    steps=STEPS, path_col="session_id", diff=("exp_group", "treatment", "control")
)
assert np.allclose(
    ss.loc[[f"funnel_cumulative_{j}" for j in range(1, 5)], "treatment"],
    [s["funnel1_conversion_rate"] for s in native["steps"]],
)

## 13. Check support before interpreting a wide or missing interval

There are only six buyers per arm in this window, each with one order. AOV cannot illustrate
the difference between personal and pooled averages here; that required the earlier independent
fixture. One order contributes much of each arm's revenue, so the large assignment N is misleading
as a description of the revenue information available.

Ratio intervals can be unavailable when a bootstrap replica has no denominator. We report that
status and its invalid-replica count. We do not discard such replicas or change B/seed to obtain
a prettier result. Explicit binary intervals address binary boundaries; they do not repair a
sparse revenue ratio or establish nominal coverage on this selected parent dataset.

In [ ]:
orders = in_window.loc[in_window.event.eq("order_success")]
support = orders.groupby("exp_group").agg(
    buyers=("user_id", "nunique"),
    orders=("user_id", "size"),
    total_revenue=("order_value_gbp", "sum"),
    largest_order=("order_value_gbp", "max"),
)
support["largest_order_revenue_share"] = support.largest_order / support.total_revenue
display(support)
display(
    ls.loc[
        ["conversion", "revenue_per_user", "average_order_value"],
        [
            "control",
            "treatment",
            "absolute_effect",
            "absolute_effect_ci_method",
            "absolute_effect_status",
            "absolute_effect_invalid_resamples",
            "relative_effect_status",
        ],
    ]
)
audit_ids = list(
    roster.loc[~roster.user_id.isin(in_window.user_id), "user_id"].head(1)
) + list(orders.user_id.unique()[:2])
lumeo_unit_metrics = lumeo_window.get_metrics(
    [ORDER, ORDERS, REVENUE], path_col="user_id"
)
lumeo_unit_metrics.columns = ["converted", "n_orders", "revenue"]
lumeo_unit_audit = roster.set_index("user_id")[["exp_group"]].join(lumeo_unit_metrics)
lumeo_unit_audit[lumeo_unit_metrics.columns] = lumeo_unit_audit[
    lumeo_unit_metrics.columns
].fillna(0.0)
display(lumeo_unit_audit.loc[audit_ids])
fig = plot_funnel(
    lumeo_result,
    "funnel_conditional",
    "Lumeo conditional transitions: sparse denominators",
)
save_figure(fig, OUT, "lumeo_sparse_funnel")
plt.show()

## 14. Keep UI completeness separate from the experiment roster

Screen rows include recorded intervals with zero taps. An incomplete interval is not a verified
zero-tap exposure. Complete zero-tap screens remain valid observations. This coverage table helps
define a subsequent UX investigation; it does not remove people from the assignment population.

The sample's three-adjacent-drawer-open pattern has no matches and is not used as a successful
pattern showcase. An absent finding should be reported as absent, not fixed by selecting a new seed.

In [ ]:
screens = lumeo["screens"]
screen_window = screens.loc[
    screens.user_id.isin(roster.user_id)
    & screens.opened_at.ge(LSTART)
    & screens.opened_at.lt(LEND)
].copy()
screen_window["complete"] = (
    screen_window.recording_complete
    & screen_window.closed_at.notna()
    & screen_window.closed_at.le(LEND)
)
screen_window["complete_zero_taps"] = screen_window.complete & screen_window.n_taps.eq(
    0
)
coverage = screen_window.groupby("screen").agg(
    recorded=("screen_instance_id", "size"),
    complete=("complete", "sum"),
    complete_zero_taps=("complete_zero_taps", "sum"),
)
display(coverage)
assert len(roster) == int(lumeo_result.diagnostics.n_assigned.sum())

## Export the calculation, not a second calculation

The same result supplies tables, plots and aggregate JSON, including interval methods, statuses,
metric definitions, population and seed. An independent validation suite checks point oracles,
session-repetition invariance, exact tiny ratio draws and boundary cases. A separate small
simulation script measures coverage/availability and test rejection with Monte Carlo uncertainty.
Those checks have different purposes; one attractive teaching realization cannot replace them.

In [ ]:
result.to_json(OUT / "demo_experiment.json")
ratios.to_json(OUT / "ratio_example.json")
boundary.to_json(OUT / "binary_boundary.json")
result.summary.to_csv(OUT / "demo_estimates.csv", index=False)
population.to_csv(OUT / "demo_population.csv")
run = {
    "retentioneering": rete.__version__,
    "demo": demo["metadata"],
    "bootstrap": BOOT,
    "binary_intervals": BINARY,
    "visual_recipe": visual.recipe(),
    "api_status": "native development branch; not PyPI 5.3.0",
}
if "lumeo" in globals():
    lumeo_result.to_json(OUT / "lumeo_experiment.json")
    lumeo_sessions.to_json(OUT / "lumeo_session_funnel.json")
    questions.to_csv(OUT / "lumeo_four_estimands.csv")
    support.to_csv(OUT / "lumeo_support.csv")
    coverage.to_csv(OUT / "lumeo_screen_coverage.csv")
    assert input_hashes == {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in (ROOT / "data" / "lumeo").glob("*")
        if p.is_file()
    }
    run["lumeo"] = {
        "selected_synthetic_parent": True,
        "input_hashes": input_hashes,
        "cohort": "assigned_at < 2026-05-11",
        "window": ["2026-05-11", "2026-05-25"],
    }
(OUT / "run_metadata.json").write_text(json.dumps(run, indent=2, default=str) + "\n")
print("Saved estimates, plots and provenance in", OUT)

## What to carry into your own analysis

Start with assignments, observation and a named estimand. Use Retentioneering paths to make
behavioral outcomes precise; keep exploratory discovery distinct from confirmatory testing.
Resample independent units with all their contributions together. Distinguish population,
path grain, denominator and interval method before interpreting a difference.

This first implementation does not support sequential monitoring, paired or cluster-randomized
assignment, individual observation horizons, or automatic causal mediation. A complete-observation
assertion must come from your logging/data contract. Meaningful product thresholds and follow-up
decisions must be stated separately; a nonsignificant result is not proof of equivalence.

Nominal confidence and calculation status are different. The independent validation audit
found substantial undercoverage for severe zero-inflated lognormal money: a nominal 95%
percentile interval covered the absolute effect in 82.8% of 2,000 unequal-arm null trials.
Increasing the bootstrap resample count did not repair that limitation. Numerical fixes
do not make heavy-tail inference automatically reliable. Read the
[validation results and assumptions](../examples/experiments/validation/RESULTS.md), inspect
outcome support, and calibrate methods for your design before using these intervals for a
product decision. The tutorial fixture is an integration example, not a coverage certificate.
